# DramaFlow Recap backend (Colab) — ဖိုင်အပြည့်အစုံပါတဲ့ version
GitHub branch မလိုဘူး၊ backend ကုဒ်အားလုံး ဒီ notebook ထဲမှာ ပါပြီးသား။ (Video Recap / Scene Recap "Auto" + Edge TTS + Self-test အတွက်)

**သုံးပုံ**
1. Colab (colab.research.google.com) > File > Upload notebook > ဒီဖိုင်ကို ရွေး
2. Cell တွေကို အစီအစဉ်အတိုင်း **တစ်ခုချင်း run** (Cell 1 → ၂ → ၃ → ၄ → ၅)
3. နောက်ဆုံး Cell ကထုတ်ပေးတဲ့ `https://….trycloudflare.com#TOKEN` တစ်ကြောင်းလုံးကို DramaFlow > Auto card > Backend ကွက်မှာ ထည့်၊ "သိမ်း + စမ်းချိတ်" နှိပ်၊ ပြီးရင် **Self-test** နှိပ်ကြည့်
4. Colab ရပ်ရင်/ပိတ်ရင် URL ပြောင်းမယ် — နောက်ဆုံး Cell ကို ပြန် run ပြီး အသစ်ယူ (ကုဒ် Cell ၂–၄ ပြန် run စရာမလို၊ runtime ပြန်စထားရင်တော့ အကုန်ပြန် run)


In [ ]:
# Cell 1: package တွေ + cloudflared တပ်ဆင် (တစ်ကြိမ်၊ ၁ မိနစ်လောက်)
!pip -q install fastapi uvicorn edge-tts google-genai imageio-ffmpeg
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
!mkdir -p /content/dramaflow_backend
print("ok")

In [ ]:
%%writefile /content/dramaflow_backend/main.py
"""DramaFlow Recap Backend (for Altare, Colab or any computer that runs Python).

Start:  uvicorn main:app --host 0.0.0.0 --port $PORT
Set the environment variable APP_TOKEN to a long secret of your own. Every call
except /health needs it in the header x-app-token. Without it a random token is
made at start and printed in the log.

What it does for the app (Video Recap and Scene Recap):
  POST /recap/jobs             start a job, answers at once with a job id
  GET  /recap/jobs/<id>        progress, then the script parts (and voice files)
  GET  /recap/jobs/<id>/audio/<n>.wav   the voice of part n (24 kHz mono WAV)
The Gemini key arrives with the request, is used for the job and is not stored.
"""
import asyncio
import base64
import json
import os
import re
import secrets
import shutil
import subprocess
import tempfile
import threading
import time
import wave
from pathlib import Path
from typing import List, Optional

import edge_tts
from fastapi import FastAPI, Header, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, Response
from google import genai
from google.genai import types
from pydantic import BaseModel

from recap_core import fit_tempo, gemini_hint, mask_key, run_recap
from tts_text import normalize_for_tts

app = FastAPI(title="DramaFlow Recap Backend")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

TOKEN = os.environ.get("APP_TOKEN") or secrets.token_urlsafe(18)
if not os.environ.get("APP_TOKEN"):
    print("APP_TOKEN not set. A temporary token for this run:", TOKEN, flush=True)
OUT_DIR = Path(tempfile.gettempdir()) / "dramaflow_recap"
OUT_DIR.mkdir(parents=True, exist_ok=True)
JOBS: dict = {}
LOCK = threading.Lock()
KEEP_SECONDS = 2 * 3600
DEFAULT_VOICE = "my-MM-ThihaNeural"


class Group(BaseModel):
    first: int
    last: int
    prompt: str
    jpeg_b64: str = ""  # Scene Recap: the numbered contact sheet
    start_ms: int = 0  # Video Recap: look only at this time range
    end_ms: int = 0
    part_ms: List[int] = []  # how long each part's video is (first..last), to fit the voice


class RecapRequest(BaseModel):
    mode: str = "scenes"  # "scenes" (pictures + dialogue) or "video" (a file already in Gemini)
    gemini_key: str
    gemini_model: str = "gemini-3.8-flash"
    gemini_fallback: str = ""  # optional second model, tried when the first is gone (404) or over its quota
    voice: str = ""  # empty = script only, no voice
    file_uri: str = ""  # Gemini Files API address (video mode)
    file_mime: str = "video/mp4"
    groups: List[Group]


def _auth(token: Optional[str]):
    if not token or not secrets.compare_digest(token, TOKEN):
        raise HTTPException(status_code=401, detail="Bad or missing x-app-token")


def _set(job_id: str, **kw):
    with LOCK:
        JOBS[job_id].update(kw)


def _ffmpeg() -> str:
    try:
        import imageio_ffmpeg

        return imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        return shutil.which("ffmpeg") or "ffmpeg"


FALLBACK_WHEN = ("404", "NOT_FOUND", "429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE")


def _gemini_fn(req: RecapRequest):
    client = genai.Client(api_key=req.gemini_key)

    def ask(model: str, contents: list) -> str:
        for attempt in range(3):
            try:
                r = client.models.generate_content(model=model, contents=contents, config=types.GenerateContentConfig(temperature=0.5))
                return r.text or ""
            except Exception as e:
                msg = str(e)
                busy = any(k in msg for k in ("429", "503", "RESOURCE_EXHAUSTED", "UNAVAILABLE"))
                if busy and attempt < 2:
                    time.sleep(25 * (attempt + 1))
                    continue
                raise RuntimeError(mask_key(msg, req.gemini_key))
        return ""

    def call(g: dict) -> str:
        contents: list = []
        if g.get("jpeg_b64"):
            contents.append(types.Part.from_bytes(data=base64.b64decode(g["jpeg_b64"]), mime_type="image/jpeg"))
        if req.mode == "video":
            if not req.file_uri:
                raise ValueError("video file_uri မပါဘူး")
            contents.append(
                types.Part(
                    file_data=types.FileData(file_uri=req.file_uri, mime_type=req.file_mime),
                    video_metadata=types.VideoMetadata(
                        start_offset=f"{max(0, int(g.get('start_ms', 0)) // 1000)}s",
                        end_offset=f"{max(1, -(-int(g.get('end_ms', 0)) // 1000))}s",
                    ),
                )
            )
        contents.append(g["prompt"])
        try:
            return ask(req.gemini_model, contents)
        except RuntimeError as e:
            fb = req.gemini_fallback.strip()
            if fb and fb != req.gemini_model and any(k in str(e) for k in FALLBACK_WHEN):
                print(f"model {req.gemini_model} failed, trying {fb}", flush=True)
                return ask(fb, contents)
            raise

    return call


def _speak(text: str, voice: str, mp3: Path, rate: str = "+0%"):
    """Edge TTS to an mp3 file, three tries."""
    for attempt in range(3):
        try:
            asyncio.run(edge_tts.Communicate(text, voice, rate=rate).save(str(mp3)))
            return
        except Exception:
            if attempt == 2:
                raise
            time.sleep(2 * (attempt + 1))


def _to_wav(src: Path, dst: Path, tempo: float = 1.0):
    """24 kHz mono WAV; tempo above 1.0 speeds the voice up (pitch is kept)."""
    cmd = [_ffmpeg(), "-y", "-i", str(src), "-ar", "24000", "-ac", "1"]
    if tempo > 1.001:
        cmd += ["-filter:a", f"atempo={tempo}"]
    cmd += ["-c:a", "pcm_s16le", str(dst)]
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0 or not dst.exists():
        raise RuntimeError("ffmpeg: " + (p.stderr or "")[-200:])


def _wav_ms(wav: Path) -> int:
    with wave.open(str(wav), "rb") as w:
        return int(w.getnframes() * 1000 / w.getframerate())


def _tts_fn(job_dir: Path, voice: str):
    use = voice if voice.startswith("my-MM-") else DEFAULT_VOICE

    def make(n: int, text: str, target_ms: int = 0):
        mp3, wav = job_dir / f"p{n}.mp3", job_dir / f"p{n}.wav"
        try:
            _speak(normalize_for_tts(text), use, mp3)
            _to_wav(mp3, wav)
            ms = _wav_ms(wav)
            tempo = fit_tempo(ms, target_ms)
            if tempo > 1.0:  # a little too long: speed it up, never more than 15%
                fast = job_dir / f"p{n}_fast.wav"
                _to_wav(mp3, fast, tempo)
                fast.replace(wav)
                ms = _wav_ms(wav)
        finally:
            mp3.unlink(missing_ok=True)
        return ms, wav.name

    return make


def clean_old():
    now = time.time()
    with LOCK:
        for jid, j in list(JOBS.items()):
            if j["status"] in ("done", "error") and now - j["created"] > KEEP_SECONDS:
                shutil.rmtree(OUT_DIR / jid, ignore_errors=True)
                del JOBS[jid]


def _save_job(job_id: str):
    """The finished job (text, warnings, audio addresses) is kept next to its
    audio files, so the app can fetch it again after the web process restarted."""
    try:
        with LOCK:
            data = dict(JOBS[job_id])
        (OUT_DIR / job_id / "job.json").write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
    except Exception:
        pass


def _load_job(job_id: str):
    if not re.fullmatch(r"recap_\d+_[0-9a-f]{4}", job_id):
        return None
    f = OUT_DIR / job_id / "job.json"
    try:
        data = json.loads(f.read_text(encoding="utf-8"))
    except Exception:
        return None
    if data.get("status") != "done" or time.time() - float(data.get("created", 0)) > KEEP_SECONDS:
        return None
    with LOCK:
        JOBS[job_id] = data
    return data


def run_job(job_id: str, req: RecapRequest):
    job_dir = OUT_DIR / job_id
    try:
        job_dir.mkdir(parents=True, exist_ok=True)
        _set(job_id, status="running", stage="စတင်နေတယ်", progress=2)
        result = run_recap(
            [g.model_dump() for g in req.groups],
            _gemini_fn(req),
            _tts_fn(job_dir, req.voice) if req.voice else None,
            lambda stage, pct: _set(job_id, stage=stage, progress=pct),
        )
        for p in result["parts"]:
            if p["audio_file"]:
                p["audio_url"] = f"/recap/jobs/{job_id}/audio/{p['n']}.wav"
            p.pop("audio_file")
        _set(job_id, status="done", stage="ပြီးပြီ", progress=100, parts=result["parts"], warnings=result["warnings"])
        _save_job(job_id)
    except Exception as e:  # shown in the app, with the key hidden
        _set(job_id, status="error", stage="မအောင်မြင်ဘူး", error=mask_key(str(e), req.gemini_key)[:400])
        shutil.rmtree(job_dir, ignore_errors=True)


@app.get("/health")
def health():
    return {"status": "ok", "service": "DramaFlow Recap Backend"}


@app.post("/recap/jobs")
def create_job(req: RecapRequest, x_app_token: Optional[str] = Header(default=None)):
    _auth(x_app_token)
    if not req.groups:
        raise HTTPException(status_code=400, detail="groups မပါဘူး")
    clean_old()
    with LOCK:
        if any(j["status"] in ("queued", "running") for j in JOBS.values()):
            raise HTTPException(status_code=409, detail="တခြား job တစ်ခု လုပ်နေတုန်းပါ")
        job_id = f"recap_{int(time.time())}_{secrets.token_hex(2)}"
        JOBS[job_id] = {"status": "queued", "stage": "စောင့်နေတယ်", "progress": 0, "created": time.time(),
                        "error": "", "parts": [], "warnings": []}
    threading.Thread(target=run_job, args=(job_id, req), daemon=True).start()
    return {"job_id": job_id}


@app.get("/recap/jobs/{job_id}")
def job_status(job_id: str, x_app_token: Optional[str] = Header(default=None)):
    _auth(x_app_token)
    with LOCK:
        j = JOBS.get(job_id)
    if j is None:
        j = _load_job(job_id)
    if j is None:
        raise HTTPException(status_code=404, detail="Job မတွေ့ပါ")
    with LOCK:
        return {k: v for k, v in j.items() if k != "created"}


@app.get("/recap/jobs/{job_id}/audio/{n}.wav")
def audio(job_id: str, n: int, x_app_token: Optional[str] = Header(default=None)):
    _auth(x_app_token)
    if not re.fullmatch(r"recap_\d+_[0-9a-f]{4}", job_id):
        raise HTTPException(status_code=404, detail="Audio not found")
    p = OUT_DIR / job_id / f"p{int(n)}.wav"
    if not p.exists():
        raise HTTPException(status_code=404, detail="Audio not found")
    return FileResponse(p, media_type="audio/wav", filename=p.name)


class TtsRequest(BaseModel):
    text: str
    voice: str = DEFAULT_VOICE
    rate: str = "+0%"


@app.post("/tts")
def tts_once(req: TtsRequest, x_app_token: Optional[str] = Header(default=None)):
    """One piece of text -> a WAV (24 kHz mono), for the app's Audio stage."""
    _auth(x_app_token)
    text = req.text.strip()
    if not text or len(text) > 6000:
        raise HTTPException(status_code=400, detail="စာသား မရှိဘူး/ရှည်လွန်းတယ် (၆၀၀၀ လုံးအထိ)")
    rate = req.rate if re.fullmatch(r"[+-]\d{1,3}%", req.rate) else "+0%"
    voice = req.voice if req.voice.startswith("my-MM-") else DEFAULT_VOICE
    d = OUT_DIR / f"tts_{secrets.token_hex(4)}"
    try:
        d.mkdir(parents=True, exist_ok=True)
        mp3, wav = d / "a.mp3", d / "a.wav"
        _speak(normalize_for_tts(text), voice, mp3, rate)
        _to_wav(mp3, wav)
        return Response(content=wav.read_bytes(), media_type="audio/wav")
    except Exception as e:
        raise HTTPException(status_code=502, detail=str(e)[:200])
    finally:
        shutil.rmtree(d, ignore_errors=True)


class SelfTestRequest(BaseModel):
    gemini_key: str = ""
    gemini_model: str = "gemini-3.8-flash"


@app.post("/selftest")
def selftest(req: SelfTestRequest, x_app_token: Optional[str] = Header(default=None)):
    """Checks what the recap needs, one by one, and says in Burmese what to do
    when something fails. Every check is [{id, ok, note}]; nothing is stored."""
    _auth(x_app_token)
    checks = []

    def add(cid: str, ok: bool, note: str):
        checks.append({"id": cid, "ok": ok, "note": note})

    ff = _ffmpeg()
    try:
        p = subprocess.run([ff, "-version"], capture_output=True, text=True, timeout=20)
        add("ffmpeg", p.returncode == 0, "ရတယ်" if p.returncode == 0 else "ffmpeg မပွင့်ဘူး (requirements ထဲ imageio-ffmpeg ပါလား)")
    except Exception as e:
        add("ffmpeg", False, "ffmpeg မရှိဘူး: " + str(e)[:80])

    d = OUT_DIR / f"selftest_{secrets.token_hex(3)}"
    try:
        d.mkdir(parents=True, exist_ok=True)
        _speak("မင်္ဂလာပါ", DEFAULT_VOICE, d / "t.mp3")
        _to_wav(d / "t.mp3", d / "t.wav")
        add("edge_tts", True, f"မြန်မာအသံ လုပ်လို့ရတယ် ({_wav_ms(d / 't.wav')} ms)")
    except Exception as e:
        add("edge_tts", False, "Edge TTS မရဘူး (backend က အင်တာနက်ထွက်လို့ရလား၊ ခဏနေ ပြန်စမ်း): " + str(e)[:100])
    finally:
        shutil.rmtree(d, ignore_errors=True)

    if not req.gemini_key.strip():
        add("gemini", False, "Gemini key မပါဘူး (Settings မှာ ထည့်ပါ)")
    else:
        try:
            client = genai.Client(api_key=req.gemini_key.strip())
            r = client.models.generate_content(model=req.gemini_model, contents="Reply with the single word OK.")
            add("gemini", True, f"{req.gemini_model} ဖြေတယ်: {(r.text or '').strip()[:20]}")
        except Exception as e:
            msg = mask_key(str(e), req.gemini_key)
            hint = gemini_hint(msg)
            add("gemini", False, hint + " · " + msg[:120])
    return {"checks": checks}


In [ ]:
%%writefile /content/dramaflow_backend/recap_core.py
"""Recap job logic with no web framework and no network, so it can be tested.

The app sends GROUPS. Each group already holds the prompt the app built (the
same prompt the manual flow copies to Gemini), an optional contact-sheet JPEG
and, for a whole video, the time range to look at. This module asks Gemini for
each group, reads the "=== PART n ===" parts out of the reply, and makes the
voice for each part. Gemini and the voice are passed in as functions.
"""
import re
import time
from typing import Callable, Dict, List, Optional, Tuple

PART_HEAD = re.compile(r"^[ \t]*[=\-#*\s]*PART\s*(\d+)[=\-#*\s:：]*$", re.IGNORECASE | re.MULTILINE)


def parse_reply(reply: str) -> Dict[int, str]:
    """Part number -> text. Text before the first part line is ignored."""
    text = (reply or "").replace("\r\n", "\n").replace("```", "")
    heads = list(PART_HEAD.finditer(text))
    out: Dict[int, str] = {}
    for i, h in enumerate(heads):
        n = int(h.group(1))
        end = heads[i + 1].start() if i + 1 < len(heads) else len(text)
        body = text[h.end():end].strip()
        if n >= 1 and body:
            out[n] = body
    return out


def gemini_hint(message: str) -> str:
    """What to do next, in Burmese, for a Gemini error text (used by the self-test)."""
    m, low = message, message.lower()
    if "location is not supported" in low or "user location" in low:
        return "Backend (Colab) ရှိတဲ့ နေရာကို Google က မခွင့်ပြုဘူး — Colab ရဲ့ Runtime > Disconnect and delete runtime လုပ်ပြီး အစကနေ ပြန် run ပါ"
    if "404" in m or "NOT_FOUND" in m:
        return "model နာမည် မရှိတော့ဘူး — model ကွက်မှာ gemini-3.8-flash လို နာမည်ပြောင်းပါ"
    if "429" in m or "RESOURCE_EXHAUSTED" in m:
        return "quota ပြည့်နေတယ် — ခဏစောင့်ပါ၊ aistudio.google.com/rate-limit မှာ စစ်ပါ"
    if "503" in m or "UNAVAILABLE" in m:
        return "Google ဘက် model လူသုံးများနေတယ် (ခဏဖြစ်တာ) — ခဏနေ ပြန်စမ်းပါ၊ အရန် model ထည့်ထားရင် Auto က အလိုအလျောက် ပြောင်းစမ်းမယ်"
    if "api key" in low or "400" in m or "403" in m or "PERMISSION" in m:
        return "key မှန်လား၊ Gemini API ဖွင့်ထားလား စစ်ပါ"
    return "မသိတဲ့ အမှား"


def mask_key(message: str, key: str) -> str:
    """Never send the user's API key back in an error text."""
    if key and len(key) >= 8:
        return message.replace(key, "***")
    return message


def part_targets(groups: List[dict]) -> Dict[int, int]:
    """Part number -> how long its video part is (ms), from each group's
    part_ms list (one entry per part, first..last). Missing or wrong-length
    lists give no target for that group."""
    out: Dict[int, int] = {}
    for g in groups:
        ms = g.get("part_ms") or []
        first, last = int(g["first"]), int(g["last"])
        if len(ms) != last - first + 1:
            continue
        for i, v in enumerate(ms):
            try:
                v = int(v)
            except (TypeError, ValueError):
                continue
            if v > 0:
                out[first + i] = v
    return out


GeminiFn = Callable[[dict], str]
TtsFn = Callable[[int, str, int], Tuple[int, str]]  # (part number, text, target ms or 0) -> (milliseconds, file name)
ProgressFn = Callable[[str, int], None]


MAX_SPEEDUP = 1.15  # the voice is sped up at most this much to fit its video part
LONG_WARN = 1.10  # still this much longer than the part after speeding up -> warn
SHORT_WARN = 0.5  # less than this share of the part -> warn
CONTEXT_CHARS = 300


def fit_tempo(audio_ms: int, target_ms: int, max_speed: float = MAX_SPEEDUP) -> float:
    """Speed factor (1.0 = unchanged) that makes a voice that is too long fit
    its video part. Never slows the voice down and never goes above max_speed."""
    if target_ms <= 0 or audio_ms <= 0 or audio_ms <= target_ms * 1.02:
        return 1.0
    return round(min(audio_ms / target_ms, max_speed), 3)


def fit_warning(n: int, audio_ms: int, target_ms: int) -> str:
    """A short Burmese note when the voice of part n does not fit its video
    part, or an empty string."""
    if target_ms <= 0 or audio_ms <= 0:
        return ""
    a, t = audio_ms / 1000, target_ms / 1000
    if audio_ms > target_ms * LONG_WARN:
        return f"အပိုင်း {n}: အသံ {a:.1f} စက္ကန့်၊ ဗီဒီယိုအပိုင်း {t:.1f} စက္ကန့် — အသံက ရှည်နေတယ် (script ကို တိုစေပါ)"
    if audio_ms < target_ms * SHORT_WARN:
        return f"အပိုင်း {n}: အသံ {a:.1f} စက္ကန့်၊ ဗီဒီယိုအပိုင်း {t:.1f} စက္ကန့် — အသံက တိုနေတယ် (အပိုင်းထဲ တိတ်နေမယ်)"
    return ""


def _tail(text: str, limit: int = CONTEXT_CHARS) -> str:
    t = " ".join(text.split())
    return t if len(t) <= limit else t[-limit:]


def with_context(prompt: str, previous_text: str) -> str:
    """Adds how the story stood at the end of the previous group, so names and
    events stay consistent from one request to the next."""
    if not previous_text.strip():
        return prompt
    return (
        prompt.rstrip()
        + "\n\nCONTEXT (the previous part of this recap ended like this. Do not repeat it; "
        + "only keep names and events consistent with it):\n"
        + _tail(previous_text)
        + "\n"
    )


def run_recap(
    groups: List[dict],
    gemini: GeminiFn,
    tts: Optional[TtsFn],
    progress: ProgressFn,
    gap_seconds: float = 6.0,
    sleep: Callable[[float], None] = time.sleep,
) -> dict:
    """Returns {"parts": [{"n", "text", "audio_ms", "audio_file"}], "warnings": [...]}.
    A group that fails is reported as a warning and the rest go on; when nothing
    at all came out, ValueError is raised."""
    warnings: List[str] = []
    texts: Dict[int, str] = {}
    total = max(1, len(groups))
    for gi, g in enumerate(groups):
        first, last = int(g["first"]), int(g["last"])
        progress(f"Gemini က အပိုင်း {first}–{last} ရေးနေတယ်", 5 + int(55 * gi / total))
        if texts:
            g = dict(g, prompt=with_context(g["prompt"], texts[max(texts)]))
        try:
            reply = gemini(g)
        except Exception as e:  # reported, the next group still runs
            warnings.append(f"အပိုင်း {first}–{last} မရဘူး: {str(e)[:160]}")
            continue
        got = parse_reply(reply)
        missing = []
        for n in range(first, last + 1):
            if n in got:
                texts[n] = got[n]
            else:
                missing.append(n)
        if missing:
            warnings.append("Gemini က မရေးပေးတဲ့ အပိုင်း: " + ", ".join(str(n) for n in missing))
        if gi + 1 < len(groups):
            sleep(gap_seconds)
    if not texts:
        raise ValueError("Gemini က script တစ်ပိုင်းမှ မပေးဘူး။ " + (warnings[0] if warnings else ""))
    targets = part_targets(groups)
    parts = []
    order = sorted(texts)
    for idx, n in enumerate(order):
        audio_ms, audio_file = 0, ""
        target = targets.get(n, 0)
        if tts is not None:
            progress(f"အသံ လုပ်နေတယ် {idx + 1}/{len(order)}", 62 + int(36 * idx / len(order)))
            try:
                audio_ms, audio_file = tts(n, texts[n], target)
                w = fit_warning(n, audio_ms, target)
                if w:
                    warnings.append(w)
            except Exception as e:
                warnings.append(f"အပိုင်း {n} အသံ မရဘူး: {str(e)[:120]}")
        parts.append({"n": n, "text": texts[n], "audio_ms": audio_ms, "target_ms": target, "audio_file": audio_file})
    return {"parts": parts, "warnings": warnings}


In [ ]:
%%writefile /content/dramaflow_backend/tts_text.py
"""Cleans narration text before it goes to the voice (Edge TTS).

Pure Python, no network. The goal is only to remove what a voice reads badly:
emoji, markdown marks, brackets with stage directions, and small plain numbers
(0-99), which are spelled as Burmese words. Everything else is left alone, so
the text the user reviewed is the text that is spoken.
"""
import re
import unicodedata

_UNITS = {1: "တစ်", 2: "နှစ်", 3: "သုံး", 4: "လေး", 5: "ငါး", 6: "ခြောက်", 7: "ခုနစ်", 8: "ရှစ်", 9: "ကိုး"}
_TENS = {1: "ဆယ်", 2: "နှစ်ဆယ်", 3: "သုံးဆယ်", 4: "လေးဆယ်", 5: "ငါးဆယ်", 6: "ခြောက်ဆယ်", 7: "ခုနစ်ဆယ်", 8: "ရှစ်ဆယ်", 9: "ကိုးဆယ်"}
_ZERO = "သုံည"


def number_words(n: int) -> str:
    """0..99 in Burmese words: 11 -> ဆယ့်တစ်, 25 -> နှစ်ဆယ့်ငါး."""
    if not 0 <= n <= 99:
        raise ValueError("only 0..99")
    if n == 0:
        return _ZERO
    if n < 10:
        return _UNITS[n]
    t, u = divmod(n, 10)
    if u == 0:
        return _TENS[t]
    # "ဆယ်" becomes "ဆယ့်" (asat after the dot below) when a unit follows
    return _TENS[t][:-1] + "\u1037\u103a" + _UNITS[u]


# A plain 1-2 digit whole number. Not part of a longer number, a decimal, a
# time (12:30), a date or range (3/4, 1-2) or a version (1.2.3).
_NUM = re.compile(r"(?<![\d.,:/\-])\d{1,2}(?![\d.,:/]|-\d)")
_BRACKETS = re.compile(r"[\(\[（【][^\)\]）】\n]{0,80}[\)\]）】]")
_MARKS = re.compile(r"[*_`#>~|=]+")
_SPACES = re.compile(r"[ \t ]+")


def _drop_symbols(text: str) -> str:
    out = []
    for ch in text:
        cat = unicodedata.category(ch)
        if cat in ("So", "Sk", "Cs", "Co", "Cn"):  # emoji, pictographs, private use
            continue
        if ch in "‍️​‌⁠﻿":  # joiners, variation selectors, zero width
            continue
        out.append(ch)
    return "".join(out)


def normalize_for_tts(text: str) -> str:
    """Returns the text the voice should read. Never returns an empty string
    for non-empty input: if everything would be removed, the original is kept."""
    original = (text or "").strip()
    t = original.replace("\r\n", "\n")
    t = _BRACKETS.sub(" ", t)  # (laughs) [music] and similar
    t = _drop_symbols(t)
    t = _MARKS.sub(" ", t)
    t = _NUM.sub(lambda m: number_words(int(m.group(0))), t)
    lines = [_SPACES.sub(" ", ln).strip() for ln in t.split("\n")]
    t = "\n".join(ln for ln in lines if ln)
    return t if t else original


In [ ]:
# Cell 5: server စတင် + public URL ထုတ် (ဒီ cell က ဆက်လည်ပတ်နေရမယ် — မရပ်ပါနဲ့)
import os, re, secrets, subprocess, time

TOKEN = secrets.token_urlsafe(18)
env = dict(os.environ, APP_TOKEN=TOKEN)
server = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000", "--log-level", "warning"],
    cwd="/content/dramaflow_backend", env=env,
)
time.sleep(4)
if server.poll() is not None:
    raise SystemExit("Server မတက်ဘူး — အပေါ်က error ကို ကြည့်ပါ (Cell 1 run ပြီးလား)")

proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print("Public URL ခဏစောင့်ပါ ...")
for line in proc.stdout:
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("\nApp ရဲ့ Backend ကွက်မှာ ဒီတစ်ကြောင်းလုံး ကူးထည့်ပါ:")
        print(f"{m.group(0)}#{TOKEN}")
        break
for _ in proc.stdout:  # keep the cell (and the tunnel) alive
    pass